In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# start date and end date
start_date = "2024-01-01"
end_date   = "2025-12-01"

In [0]:
# 1️⃣ Generate one row per month start between start_date and end_date
df = (
    spark.sql(f"""
        SELECT explode(
            sequence(
                to_date('{start_date}'),
                to_date('{end_date}'),
                interval 1 month
            )
        ) AS month_start_date
    """)
)


# 2️⃣ Add useful analytics columns
df = (
    df
    # Surrogate key at month grain
    .withColumn("date_key", date_format("month_start_date", "yyyyMM").cast("int"))
    .withColumn("year", year("month_start_date"))
    .withColumn("month_name", date_format("month_start_date", "MMMM"))
    .withColumn("month_short_name", date_format("month_start_date", "MMM"))
    .withColumn("quarter", concat(lit("Q"), quarter("month_start_date")))
    .withColumn("year_quarter", concat(col("year"), lit("-Q"), quarter("month_start_date")))
)

In [0]:
display(df)

In [0]:
# Save as a table
df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("fmcg.gold.dim_date")